In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", '')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", ''):
        sh(os.environ.get("PIP_POST", ''))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    's1_final.py': '"""S1 of llm_stages_v1: final T1 claim detectors for the product, and their reading of the GS-Eval sentences.\n\n    python -u s1_final.py            (Kaggle T4; inputs in /kaggle/input/gs-llm-stages-v1 and /kaggle/input/greenscan-t1-data)\n\nRecipe = training/kaggle/t1_kaggle.py (HP e5-small / e5-base, MAX_LEN 160, batch 16, masked text, "query: " prefix),\ntrained on every T1 training sentence EXCEPT those whose first 60 normalised characters match a GS-Eval sentence\n(registration llm_stages_v1: 16 overlap). Two seeds per model, probabilities averaged.\nWrites out/s1_<model>.jsonl ({id, probs, label}) for the 200 GS-Eval sentences and out/<model>-final/ (weights +\ntokenizer, seed 13) for the product (src/quantum_gw/agents/claim_model.py).\n"""\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\n\nLABELS = ["NOT_CLAIM", "QUALITATIVE_CLAIM", "QUANTIFIED_CLAIM", "VAGUE"]\nL2I = {l: i for i, l in enumerate(LABELS)}\nMODELS = {"e5-small": "intfloat/multilingual-e5-small", "e5-base": "intfloat/multilingual-e5-base"}\nHP = {"e5-small": dict(lr=5e-5, epochs=6), "e5-base": dict(lr=3e-5, epochs=6)}\nMAX_LEN, BATCH, SEEDS = 160, 16, [13, 29]\nDEV = "cuda" if torch.cuda.is_available() else "cpu"\n\n\ndef find(name):\n    for p in Path("/kaggle/input").rglob(name):\n        return p\n    return Path(name)\n\n\ndef norm(s):\n    return re.sub(r"\\s+", " ", s.lower()).strip()\n\n\ndef enc(tok, texts):\n    return tok(["query: " + t for t in texts], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")\n\n\ndef fit(key, rows, seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    tok = AutoTokenizer.from_pretrained(MODELS[key])\n    model = AutoModelForSequenceClassification.from_pretrained(\n        MODELS[key], num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I).to(DEV)\n    hp = HP[key]\n    steps = hp["epochs"] * ((len(rows) + BATCH - 1) // BATCH)\n    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=0.01)\n    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)\n    scaler = torch.amp.GradScaler(enabled=DEV == "cuda")\n    model.train()\n    for ep in range(hp["epochs"]):\n        order = list(range(len(rows)))\n        random.shuffle(order)\n        total = 0.0\n        for i in range(0, len(order), BATCH):\n            b = [rows[j] for j in order[i:i + BATCH]]\n            x = {k: v.to(DEV) for k, v in enc(tok, [r["text_masked"] for r in b]).items()}\n            y = torch.tensor([L2I[r["label"]] for r in b], device=DEV)\n            with torch.autocast(DEV, enabled=DEV == "cuda"):\n                loss = model(**x, labels=y).loss\n            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()\n            total += loss.item()\n        print(f"[s1] {key} seed {seed} epoch {ep + 1}/{hp[\'epochs\']} loss {total:.2f}", flush=True)\n    return model.eval(), tok\n\n\n@torch.no_grad()\ndef probs(model, tok, texts):\n    out = []\n    for i in range(0, len(texts), 64):\n        x = {k: v.to(DEV) for k, v in enc(tok, texts[i:i + 64]).items()}\n        out += torch.softmax(model(**x).logits.float(), -1).cpu().tolist()\n    return out\n\n\ndef main():\n    print("[cell] s1 load data", flush=True)\n    train = [json.loads(x) for x in open(find("train.jsonl"), encoding="utf-8")]\n    ev = [json.loads(x) for x in open(find("s1_eval.jsonl"), encoding="utf-8")]\n    if os.environ.get("LIMIT"):\n        train = train[: int(os.environ["LIMIT"])]\n    held = {r["norm60"] for r in ev}\n    kept = [r for r in train if norm(r["text"])[:60] not in held and r.get("label") in L2I]\n    print(f"[s1] train {len(train)} -> {len(kept)} after removing GS-Eval overlap; eval {len(ev)}", flush=True)\n    Path("out").mkdir(exist_ok=True)\n    summary = {"train_rows": len(kept), "removed_overlap": len(train) - len(kept), "models": {}}\n    for key in MODELS:\n        t0, acc = time.time(), None\n        for seed in SEEDS:\n            model, tok = fit(key, kept, seed)\n            p = np.array(probs(model, tok, [r["text_masked"] for r in ev]))\n            acc = p if acc is None else acc + p\n            if seed == SEEDS[0]:\n                model.save_pretrained(f"out/{key}-final"); tok.save_pretrained(f"out/{key}-final")\n            del model\n            torch.cuda.empty_cache()\n        acc /= len(SEEDS)\n        with open(f"out/s1_{key}.jsonl", "w", encoding="utf-8") as fh:\n            for r, pr in zip(ev, acc.tolist()):\n                fh.write(json.dumps({"id": r["id"], "probs": dict(zip(LABELS, [round(x, 4) for x in pr])),\n                                     "label": LABELS[int(np.argmax(pr))]}) + "\\n")\n        summary["models"][key] = {"seconds": round(time.time() - t0)}\n        print(f"[s1] {key} done in {time.time() - t0:.0f}s", flush=True)\n    json.dump(summary, open("out/s1_summary.json", "w"))\n    print("[s1] SUMMARY", json.dumps(summary), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u s1_final.py', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u s1_final.py')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)